<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2014%20-%202026-05-12%20-%20Dashboards%20con%20Streamlit/clase_14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 14 · Dashboards con Streamlit

Hasta ahora todo lo que has construido vive en un notebook o un script Python. Nadie más lo ve. Si quieres compartir el análisis con un jefe, un cliente o un compañero de otro equipo, mandas el notebook por correo o le dices "instala pandas y corre esto". Esperemos que tengan Python.

Hoy cambia eso. Conviertes el análisis en una app web con Streamlit — un framework que entiende que los analistas no somos ingenieros web. Escribís Python, y en 5 minutos tenés una interfaz interactiva con filtros, KPIs, gráficos, todo en un navegador. Si quieres, la despliegas gratis en Streamlit Cloud y le das un link a cualquiera.

> **Hoy haces** · Construir un dashboard en Streamlit: layout, sidebar con filtros, métricas, al menos un gráfico interactivo y una tabla. ~2h30.
>
> **Entrega** · El archivo `app.py` funcionando localmente con `streamlit run app.py`, al repo del equipo antes de la próxima clase.

In [ ]:
# --- Setup del entorno
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Estilo visual
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["figure.dpi"] = 100

print("Setup completo ✓")

## Por qué Streamlit existe

En 2019 Adrien Treuille y Thiago Teixeira estaban frustrados. Habían entrenado un modelo increíble, pero mostrárselo a otros requería aprender Flask, HTML, CSS, JavaScript, desplegar un servidor. O simplemente mandar un notebook por correo esperando que el destinatario tuviera Python instalado.

Streamlit fue la respuesta: **cualquier script Python se convierte en una app web escribiendo solo Python**. Sin HTML. Sin JavaScript. Sin servidor complicado. En 2022 Snowflake lo compró por 800 millones de dólares porque funciona tan bien que ya está en todas partes.

La clave está en una idea simple: cada vez que alguien interactúa con un widget (un slider, un selector de fechas), Streamlit **re-ejecuta el script entero de arriba a abajo**. Parece ineficiente, pero simplifica el modelo mental enormemente. No hay callbacks, no hay estado complejo, no hay dos versiones de la verdad. Es como un cuaderno vivo.

## Los widgets más comunes

| Widget | Para qué |
|---|---|
| `st.title()`, `st.header()`, `st.write()` | Texto y encabezados |
| `st.metric()` | Números grandes (KPIs) |
| `st.sidebar.*` | Construir un panel lateral con filtros |
| `st.selectbox()`, `st.multiselect()` | Dropdowns y selección múltiple |
| `st.slider()`, `st.date_input()` | Rangos y fechas |
| `st.columns()` | Dividir en columnas el layout |
| `st.tabs()` | Pestañas para organizar secciones |
| `st.plotly_chart()`, `st.pyplot()` | Gráficos interactivos o estáticos |
| `st.dataframe()` | Tablas que el usuario puede ordenar |
| `st.map()` | Mapa simple con coordenadas |

Una cosa importante: `@st.cache_data`. Si cargás un archivo grande o hacés un cálculo pesado, decoralo con eso y Streamlit lo cachea. La próxima vez que se ejecute el script, no vuelve a hacerlo. Solo al principio.

## Un dashboard mínimo

Vamos a construir una app que carga el Titanic, muestra KPIs filtrados por clase y sexo, y un par de gráficos. Copiá esto en un archivo llamado `app.py`:

In [ ]:
CODIGO_DASHBOARD = '''
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title='Análisis Titanic', page_icon='🚢', layout='wide')

@st.cache_data
def cargar_datos():
    url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
    return pd.read_csv(url)

df = cargar_datos()

st.title('🚢 Análisis del Titanic')
st.caption('Dashboard demostrativo — seminario EDA')

with st.sidebar:
    st.header('Filtros')
    clase = st.multiselect('Clase', sorted(df['Pclass'].unique()), default=sorted(df['Pclass'].unique()))
    sexo = st.multiselect('Sexo', df['Sex'].unique(), default=list(df['Sex'].unique()))

f = df[df['Pclass'].isin(clase) & df['Sex'].isin(sexo)]

c1, c2, c3 = st.columns(3)
c1.metric('Pasajeros', f\"{len(f):,}\")
c2.metric('% Supervivencia', f\"{f['Survived'].mean() * 100:.1f}%\")
c3.metric('Edad media', f\"{f['Age'].mean():.1f}\")

st.divider()

tab1, tab2 = st.tabs(['Demografía', 'Supervivencia'])

with tab1:
    st.plotly_chart(
        px.histogram(f, x='Age', color='Sex', nbins=30, barmode='overlay', title='Distribución de edad'),
        use_container_width=True,
    )

with tab2:
    g = f.groupby(['Pclass', 'Sex'])['Survived'].mean().reset_index()
    st.plotly_chart(
        px.bar(g, x='Pclass', y='Survived', color='Sex', barmode='group', title='% supervivencia por clase y sexo'),
        use_container_width=True,
    )
    with st.expander('Ver datos filtrados'):
        st.dataframe(f)
'''
print(CODIGO_DASHBOARD)

Guardá ese string en `app.py` en tu carpeta de proyecto. Luego en la terminal:

```bash
pip install streamlit plotly
streamlit run app.py
```

Se abre automáticamente en `http://localhost:8501`. Cada vez que guardas el archivo, Streamlit te ofrece recargar.

¿Qué pasó? Streamlit leyó tu script, vio los widgets y generó una interfaz. Cuando movés un slider, el script se ejecuta de nuevo con los nuevos valores. Es reactivo, pero sin callbacks complicados.

¿Es ineficiente re-ejecutar todo? Algo. ¿Es el trade-off más importante? Sí. El código es **tan legible** que cualquiera puede entenderlo y modificarlo seis meses después.

¿Qué pasa si alguien interactúa con dos widgets a la vez? Digamos que seleccionan una clase diferente Y mueven el slider de edad al mismo tiempo. Streamlit re-ejecuta el script una sola vez, con los dos valores nuevos. El modelo mental es más limpio que con callbacks tradicionales.

¿La carga se vuelve lenta? A veces, especialmente si el dataset es enorme. Por eso existe `@st.cache_data` — cachea el resultado de una función. Si llamas `cargar_datos()` una vez, Streamlit lo cachea. Las próximas veces devuelve el resultado sin re-ejecutar la función.

## Arquitectura: separar datos, lógica y UI

Un dashboard profesional separa responsabilidades en tres archivos:

1. **`data.py`** — funciones que cargan, limpian, cachean. Solo I/O.
2. **`features.py`** — cálculos, transformaciones, lógica. Sin UI.
3. **`app.py`** — solo widgets y llamadas a las otras dos.

Así si mañana necesitas los mismos KPIs en un reporte automatizado o una API, importás desde `features.py` sin copiar código. Es modularidad básica pero muchos dashboards la ignoran.

## Despliegue gratuito en Streamlit Cloud

1. Empujá el código a un repo público en GitHub (incluyendo `requirements.txt` con las dependencias).
2. Entré a https://streamlit.io/cloud y logueate con GitHub.
3. "New app" → seleccioná el repo, rama y archivo (`app.py`).
4. Presioná deploy.
5. En 2 minutos tenés una URL del tipo `https://<tu-app>.streamlit.app`.

Es tan simple que muchas personas montan sus análisis como dashboards en Streamlit Cloud para que los vea el mundo. Gratis, sin servidor propio.

## Checkpoint

Antes de terminar:

1. **Streamlit re-ejecuta el script** cada vez que hay interacción, por lo que el modelo mental es simple (no callbacks).
2. **`@st.cache_data`** cachea funciones para no recalcular innecesariamente.
3. **Layouts con `st.columns()` y `st.tabs()`** para organizar visualmente.
4. **KPIs con `st.metric()`** destacan números importantes.
5. **Despliegue en Streamlit Cloud** es gratuito y automático desde un repo Git.

Si algo no quedó claro, ahora es mejor preguntar que después luchar contra un dashboard que no funciona.

In [ ]:
# Validación
assert 'CODIGO_DASHBOARD' in dir(), 'Deberías haber visto el código'
print("✓ Checkpoint completado")

## El flujo del dashboard

```
Datos crudos (CSV, BD)
       ↓
  cargar_datos() con @st.cache_data
       ↓
  Filtros en sidebar
       ↓
  Dataset filtrado
       ↓
  KPIs → st.metric()
       ↓
  Gráficos → st.plotly_chart()
       ↓
  Usuario ve app en navegador
```

## Patrones que funcionan

- **KPIs grandes arriba, gráficos abajo.** Los usuarios escanean de arriba hacia abajo.
- **Filtros a la izquierda en la sidebar.** Es donde esperan encontrar controles.
- **Un dashboard, un propósito.** No intentes meter 10 reportes en la misma app.
- **Usar tabs para historias relacionadas.** "Demografía", "Supervivencia", etc.
- **Expandibles para detalles.** Con `st.expander()` ocultás datos que pocos necesitan ver.
- **Títulos afirmativos en gráficos.** No "Supervivencia por sexo". Sí "El 76% de las mujeres sobrevivió".

## Para tu equipo

Construyan un dashboard con:
- Al menos 3 KPIs relevantes para su caso.
- 2 gráficos filtrables (usando selectores en la sidebar).
- Sidebar limpia con 2-3 filtros máximo.
- Una tabla expandible con datos filtrados.

El entregable es la carpeta `app/` con `app.py` funcional. Idealmente desplegado en Streamlit Cloud.

> **Recordatorio** · `app.py` funcional al repo antes de la próxima sesión.

---

Hoy convertiste código Python en una app web interactiva sin escribir HTML, CSS ni JavaScript. Eso es lo que hace Streamlit valioso: **reduce el overhead técnico para que los analistas se concentren en la lógica**.

Mañana vemos cómo servir un modelo entrenado como API REST. El dashboard es para humanos. La API es para máquinas. Juntas forman un entregable profesional completo.